Read in packages. Create df

In [24]:
import nflreadpy as nfl
import pandas as pd

seasons = list(range(2016,2026))
pbp = nfl.load_pbp(seasons=seasons).to_pandas()

columns_to_keep = [
    "game_id", "season", "week", "game_date",
    "home_team", "away_team", "posteam", "posteam_type", "defteam",
    "play_id", "order_sequence", "play_type", "timeout_team", "desc",
    "kicker_player_id", "kicker_player_name",
    "kick_distance", "field_goal_result",
    "qtr", "game_seconds_remaining", "score_differential",
    "temp", "wind", "roof", "weather", "half_seconds_remaining", "game_half",
]
columns_to_keep = [col for col in columns_to_keep if col in pbp.columns]
pbp = pbp.select(columns_to_keep)

sort_cols = ["game_id", "play_id"]
if "order_sequence" in pbp.columns and pbp["order_sequence"].notna().any():
    sort_cols = ["game_id", "order_sequence", "play_id"]
pbp = pbp.sort_values(sort_cols, kind="stable").copy()

pbp["previous_timeout_team"] = (
    pbp.groupby("game_id")["timeout_team"].shift(1)
)
pbp["previous_play_description"] = (
    pbp.groupby("game_id")["desc"].shift(1)
)

create fg dataframe

In [25]:
fg = pbp.loc[pbp["play_type"].eq("field_goal")].copy()
fg["defensive_timeout_before_kick"] = (
    fg["previous_timeout_team"].eq(fg["defteam"])
)
fg["made"] = fg["field_goal_result"].eq("made")

print(fg.head())

             game_id  season  week   game_date home_team away_team posteam  \
34   2016_01_BUF_BAL    2016     1  2016-09-11       BAL       BUF     BAL   
93   2016_01_BUF_BAL    2016     1  2016-09-11       BAL       BUF     BUF   
139  2016_01_BUF_BAL    2016     1  2016-09-11       BAL       BUF     BAL   
237  2016_01_CAR_DEN    2016     1  2016-09-08       DEN       CAR     CAR   
303  2016_01_CAR_DEN    2016     1  2016-09-08       DEN       CAR     CAR   

    posteam_type defteam  play_id  ...  temp  wind      roof  \
34          home     BUF    798.0  ...  84.0   8.0  outdoors   
93          away     BAL   2119.0  ...  84.0   8.0  outdoors   
139         home     BUF   3214.0  ...  84.0   8.0  outdoors   
237         away     DEN   1826.0  ...  85.0  10.0  outdoors   
303         away     DEN   3425.0  ...  85.0  10.0  outdoors   

                                               weather half_seconds_remaining  \
34   Partly Cloudy Temp: 84° F, Humidity: 44%, Wind...           

check data

In [27]:
overall = (
    fg.groupby("defensive_timeout_before_kick")
    .agg(
        attempts=("made", "size"),
        made=("made", "sum"),
        make_rate=("made", "mean"),
    )
)

print(overall)


print(
    fg.loc[
        fg["defensive_timeout_before_kick"],
        [
            "game_id", "game_date", "posteam", "defteam",
            "kick_distance", "field_goal_result",
            "previous_timeout_team", "previous_play_description",
        ],
    ].head(20).to_string(index=False)
)

                               attempts  made  make_rate
defensive_timeout_before_kick                           
False                             10031  8539   0.851261
True                                700   546   0.780000
        game_id  game_date posteam defteam  kick_distance field_goal_result previous_timeout_team                                                                                                                                                                                                   previous_play_description
2016_01_CAR_DEN 2016-09-08     CAR     DEN           50.0            missed                   DEN                                                                                                                                                                                                 Timeout #2 by DEN at 00:09.
2016_01_CIN_NYJ 2016-09-11     CIN     NYJ           47.0              made                   NYJ                                       

compare to similar kicks

In [28]:
fg["distance_band"] = pd.cut(
    fg["kick_distance"],
    bins=[0, 30, 40, 50, 60, 100],
)

distance_home_away = (
    fg.groupby(
        ["distance_band", "posteam_type", "defensive_timeout_before_kick"],
        observed=True,
    )
    .agg(
        attempts=("made", "size"),
        make_rate=("made", "mean"),
    )
    .reset_index()
)

print(distance_home_away)

   distance_band posteam_type  defensive_timeout_before_kick  attempts  \
0        (0, 30]         away                          False      1308   
1        (0, 30]         away                           True        51   
2        (0, 30]         home                          False      1333   
3        (0, 30]         home                           True        62   
4       (30, 40]         away                          False      1442   
5       (30, 40]         away                           True        92   
6       (30, 40]         home                          False      1437   
7       (30, 40]         home                           True       104   
8       (40, 50]         away                          False      1443   
9       (40, 50]         away                           True       109   
10      (40, 50]         home                          False      1511   
11      (40, 50]         home                           True       110   
12      (50, 60]         away         

compare to more variables

In [29]:
import numpy as np

# Make a separate, complete analysis table with one row per field goal attempt.
fg_dataset = fg.copy()

fg_dataset["half_seconds_remaining"] = pd.to_numeric(
    fg_dataset["half_seconds_remaining"], errors="coerce"
)

in_regulation_half = fg_dataset["game_half"].isin(["Half1", "Half2"])

fg_dataset["half_end_group"] = np.select(
    [
        in_regulation_half & (fg_dataset["half_seconds_remaining"] <= 120),
        in_regulation_half & (fg_dataset["half_seconds_remaining"] > 120),
        fg_dataset["game_half"].eq("Overtime"),
    ],
    [
        "Final 2 minutes of half",
        "More than 2 minutes from half end",
        "Overtime",
    ],
    default="Unknown",
)

# Make sure the variables used to create groups are numeric.
fg_dataset["kick_distance"] = pd.to_numeric(
    fg_dataset["kick_distance"], errors="coerce"
)
fg_dataset["game_seconds_remaining"] = pd.to_numeric(
    fg_dataset["game_seconds_remaining"], errors="coerce"
)
fg_dataset["score_differential"] = pd.to_numeric(
    fg_dataset["score_differential"], errors="coerce"
)
fg_dataset["wind"] = pd.to_numeric(fg_dataset["wind"], errors="coerce")

# Group kicks by distance.
fg_dataset["distance_band"] = pd.cut(
    fg_dataset["kick_distance"],
    bins=[0, 30, 40, 50, 60, 100],
    labels=["30 yards or less", "31–40", "41–50", "51–60", "Over 60"],
    include_lowest=True,
)

# Group kicks by time remaining in the game.
fg_dataset["game_time_group"] = pd.cut(
    fg_dataset["game_seconds_remaining"],
    bins=[-1, 120, 600, 1800, np.inf],
    labels=["Final 2 minutes", "2–10 minutes", "10–30 minutes", "Over 30 minutes"],
)

# Describe the score from the kicking team's perspective.
score = fg_dataset["score_differential"]
fg_dataset["score_state"] = np.select(
    [
        score <= -8,
        (score >= -7) & (score < 0),
        score == 0,
        (score > 0) & (score <= 7),
        score > 7,
    ],
    ["Behind by 8+", "Behind by 1–7", "Tied", "Ahead by 1–7", "Ahead by 8+"],
    default="Unknown",
)

# Use 15+ mph as an exploratory strong-wind category.
fg_dataset["wind_group"] = pd.cut(
    fg_dataset["wind"],
    bins=[-np.inf, 9, 14, np.inf],
    labels=["0–9 mph", "10–14 mph", "15+ mph"],
).astype("object").fillna("Unknown")

# Overall comparison: timeout immediately before the kick vs. no timeout.
overall_summary = (
    fg_dataset.groupby("defensive_timeout_before_kick")
    .agg(
        attempts=("made", "size"),
        makes=("made", "sum"),
        make_rate=("made", "mean"),
    )
    .reset_index()
)
overall_summary["make_rate_percent"] = overall_summary["make_rate"] * 100

print("Overall make rates:")
print(overall_summary.to_string(index=False))

Overall make rates:
 defensive_timeout_before_kick  attempts  makes  make_rate  make_rate_percent
                         False     10031   8539   0.851261          85.126109
                          True       700    546   0.780000          78.000000


summary table

In [30]:
summary_parts = []

for column, label in [
    ("distance_band", "Kick distance"),
    ("posteam_type", "Kicking team location"),
    ("game_time_group", "Game time remaining"),
    ("half_end_group", "Time until end of half"),
    ("score_state", "Score state"),
    ("wind_group", "Wind"),
    ("roof", "Stadium roof"),
]:
    if column in fg_dataset.columns:
        part = (
            fg_dataset
            .groupby(
                [column, "defensive_timeout_before_kick"],
                dropna=False,
                observed=True,
            )
            .agg(
                attempts=("made", "size"),
                makes=("made", "sum"),
                make_rate=("made", "mean"),
            )
            .reset_index()
            .rename(columns={column: "category"})
        )
        part.insert(0, "feature", label)
        part["make_rate_percent"] = part["make_rate"] * 100
        summary_parts.append(part)

summary_table = pd.concat(summary_parts, ignore_index=True)

print("\nMake rates by comparison group:")
print(summary_table.to_string(index=False))


Make rates by comparison group:
               feature                          category  defensive_timeout_before_kick  attempts  makes  make_rate  make_rate_percent
         Kick distance                  30 yards or less                          False      2641   2582   0.977660          97.765998
         Kick distance                  30 yards or less                           True       113    108   0.955752          95.575221
         Kick distance                             31–40                          False      2879   2638   0.916290          91.629038
         Kick distance                             31–40                           True       196    179   0.913265          91.326531
         Kick distance                             41–50                          False      2954   2299   0.778267          77.826676
         Kick distance                             41–50                           True       219    154   0.703196          70.319635
         Kick distance

match kicks comparison

In [34]:
match_columns = [
    "kicker_player_id",
    "kick_distance",
    "posteam_type",
    "half_end_group",
    "score_state",
    "wind_group",
    "roof",
]

# Remove kicks that cannot be matched on kicker and distance.
match_data = fg_dataset.dropna(
    subset=["kicker_player_id", "kick_distance"]
).copy()

strata_rates = (
    match_data
    .groupby(
        match_columns + ["defensive_timeout_before_kick"],
        dropna=False,
        observed=True,
    )
    .agg(
        attempts=("made", "size"),
        make_rate=("made", "mean"),
    )
    .reset_index()
)

matched_table = strata_rates.pivot(
    index=match_columns,
    columns="defensive_timeout_before_kick",
    values=["attempts", "make_rate"],
)

# Give the columns readable names.
matched_table.columns = [
    f"{measure}_{'timeout' if timeout else 'no_timeout'}"
    for measure, timeout in matched_table.columns
]
matched_table = matched_table.reset_index()

# Keep only groups that have both types of kick.
matched_table = matched_table.dropna(
    subset=["make_rate_timeout", "make_rate_no_timeout"]
)
matched_table["make_rate_difference_percentage_points"] = (
    matched_table["make_rate_timeout"]
    - matched_table["make_rate_no_timeout"]
) * 100

print("\nStrictly matched groups:")
print(f"Number of groups with both kick types: {len(matched_table)}")
print(matched_table.to_string(index=False))


Strictly matched groups:
Number of groups with both kick types: 42
kicker_player_id  kick_distance posteam_type                    half_end_group   score_state wind_group     roof  attempts_no_timeout  attempts_timeout  make_rate_no_timeout  make_rate_timeout  make_rate_difference_percentage_points
      00-0020578           33.0         home More than 2 minutes from half end   Ahead by 8+    Unknown   closed                  1.0               1.0                   1.0                1.0                                     0.0
      00-0023252           41.0         away More than 2 minutes from half end  Ahead by 1–7    Unknown   closed                  1.0               1.0                   0.0                1.0                                   100.0
      00-0023853           43.0         home More than 2 minutes from half end  Ahead by 1–7    Unknown   closed                  1.0               1.0                   1.0                0.0                                  -100.0


totals

In [36]:
negative_differences = (
    matched_table.loc[
        matched_table["make_rate_difference_percentage_points"] < 0
    ]
    .sort_values("make_rate_difference_percentage_points")
)

print(negative_differences.to_string(index=False))

differences = matched_table["make_rate_difference_percentage_points"]

below_zero = (differences < 0).sum()
zero_or_above = (differences >= 0).sum()
total = len(matched_table)

print(f"Below 0: {below_zero} of {total} ({below_zero / total:.1%})")
print(f"Equal to or above 0: {zero_or_above} of {total} ({zero_or_above / total:.1%})")

by_timeout = overall_summary.set_index("defensive_timeout_before_kick")

timeout_row = by_timeout.loc[True]
no_timeout_row = by_timeout.loc[False]

total_attempts = timeout_row["attempts"] + no_timeout_row["attempts"]
timeout_share = timeout_row["attempts"] / total_attempts * 100
rate_difference = (
    timeout_row["make_rate_percent"] - no_timeout_row["make_rate_percent"]
)

print(
    f"Of {total_attempts:,} field goal attempts, "
    f"{timeout_row['attempts']:,} ({timeout_share:.1f}%) had a defensive "
    f"timeout immediately before the kick. The make rate was "
    f"{timeout_row['make_rate_percent']:.1f}% for those kicks and "
    f"{no_timeout_row['make_rate_percent']:.1f}% for other kicks, "
    f"a difference of {rate_difference:.1f} percentage points."
)

kicker_player_id  kick_distance posteam_type                    half_end_group   score_state wind_group     roof  attempts_no_timeout  attempts_timeout  make_rate_no_timeout  make_rate_timeout  make_rate_difference_percentage_points
      00-0023853           43.0         home More than 2 minutes from half end  Ahead by 1–7    Unknown   closed                  1.0               1.0                   1.0                0.0                                  -100.0
      00-0025580           39.0         home           Final 2 minutes of half  Ahead by 1–7  10–14 mph outdoors                  1.0               1.0                   1.0                0.0                                  -100.0
      00-0031136           45.0         away More than 2 minutes from half end Behind by 1–7    0–9 mph outdoors                  1.0               1.0                   1.0                0.0                                  -100.0
      00-0031492           61.0         away           Final 2 minut